# 强化学习算法总结与对比

这份笔记用于梳理 RL 文件夹中出现的主要算法与方法，重点回答四个问题：

1. 这些方法的基本结构是什么。
2. 它们想优化的目标是什么。
3. 它们各自适合什么问题。
4. 它们之间的关键差异是什么。

## 一条主线看完整个 RL 体系

可以把这些内容看成从“如何估计价值”到“如何直接优化策略”，再到“如何利用模型、数据和多智能体结构”的演化过程：

- 基础问题建模：多臂老虎机、马尔可夫决策过程 MDP
- 经典值函数方法：动态规划 DP、蒙特卡洛 MC、时序差分 TD、Dyna-Q
- 深度值函数方法：DQN 及其改进
- 策略优化方法：策略梯度、Actor-Critic、TRPO、PPO
- 连续控制方法：DDPG、SAC
- 模仿与逆强化学习：Behavior Cloning、IRL
- 基于模型方法：MPC、基于模型的策略优化
- 特殊设定：离线强化学习、目标导向强化学习、多智能体强化学习、MCTS

## 对比时的几个核心维度

后面的总结和表格会反复使用以下维度：

- 结构：算法内部由哪些模块组成，例如 value、policy、model、buffer。
- 目标：算法实际在优化什么，例如期望回报、Bellman 误差、策略目标、熵正则目标。
- 特性：样本效率、训练稳定性、是否适合连续动作、是否依赖环境模型、是否支持离线数据。

## 基础与值函数方法

### 1. 多臂老虎机
- 结构：只有动作选择器和奖励估计器，没有状态转移。
- 目标：在探索与利用之间平衡，使累计奖励最大。
- 特性：问题最简单，常作为 RL 中探索机制的起点；典型方法包括 $\epsilon$-greedy、UCB、Thompson Sampling。

### 2. 马尔可夫决策过程 MDP
- 结构：状态 $s$、动作 $a$、转移概率 $P$、奖励函数 $R$、折扣因子 $\gamma$。
- 目标：定义强化学习问题，使策略 $\pi(a|s)$ 的长期累计回报最大。
- 特性：它本身不是具体算法，而是大多数 RL 算法的统一数学框架。

### 3. 动态规划 DP
- 结构：基于已知环境模型，通过策略评估和策略改进迭代求解。
- 目标：求出最优价值函数 $V^*$ 或 $Q^*$，进而得到最优策略。
- 特性：依赖精确模型；理论清晰，但真实复杂环境通常无法直接使用。

### 4. 蒙特卡洛 MC
- 结构：通过完整轨迹采样，用回合回报直接估计价值。
- 目标：用采样均值逼近状态价值或动作价值。
- 特性：不需要环境模型；无偏，但方差较大，必须等到回合结束后更新。

### 5. 时序差分 TD
- 结构：使用一步 bootstrap 更新，例如 TD(0)、SARSA、Q-learning。
- 目标：利用 Bellman 方程递推估计价值函数。
- 特性：能在线学习，方差低于 MC；但会引入偏差。Q-learning 是 off-policy，SARSA 是 on-policy。

### 6. Dyna-Q
- 结构：真实环境学习 + 学到的模型回放规划，两部分交替进行。
- 目标：同时利用真实交互和模型模拟，提高样本效率。
- 特性：把 model-free 和 model-based 结合起来；当模型足够准时，学习速度明显更快。

### 7. DQN
- 结构：Q 网络 + 经验回放 + 目标网络。
- 目标：最小化 TD 目标误差，学习近似最优动作价值函数。
- 特性：解决离散动作空间下高维输入问题；是深度强化学习的重要起点，但训练可能不稳定。

### 8. DQN 改进算法
- 结构：在 DQN 基础上加入 Double DQN、Dueling DQN、Prioritized Replay 等模块。
- 目标：降低 Q 值高估、提升价值表达能力、提高采样利用率。
- 特性：
  - Double DQN：缓解最大化带来的过估计。
  - Dueling DQN：分离状态价值与动作优势。
  - PER：优先学习 TD 误差较大的样本。

### 9. MCTS
- 结构：选择、扩展、模拟、回传四步循环构成搜索树。
- 目标：通过前瞻搜索逼近更优决策。
- 特性：更像在线规划方法，适合可模拟环境和搜索空间明确的问题，例如博弈和规划。

## 策略优化与连续控制方法

### 10. 策略梯度 PG
- 结构：直接参数化策略 $\pi_\theta(a|s)$，通过采样估计梯度更新参数。
- 目标：最大化期望累计回报 $J(\theta)$。
- 特性：可以直接处理连续动作；但梯度方差较大，样本效率通常不高。

### 11. Actor-Critic
- 结构：Actor 负责输出动作策略，Critic 负责估计价值或优势函数。
- 目标：Actor 用 Critic 提供的信号更新策略，Critic 学习更准确的价值估计。
- 特性：兼具策略梯度的灵活性与值函数方法的低方差；是很多现代算法的基础模板。

### 12. TRPO
- 结构：Actor-Critic 框架 + 信赖域约束。
- 目标：在每次更新时限制新旧策略差异，保证策略性能不因步子过大而崩掉。
- 特性：训练更稳，理论更强；但实现复杂，需要二阶近似或共轭梯度等技巧。

### 13. PPO
- 结构：Actor-Critic 框架 + clipped surrogate objective。
- 目标：在简化 TRPO 的同时，限制策略更新幅度并稳定训练。
- 特性：实现简单、效果稳健、应用广泛，是工业和研究里最常见的 on-policy 算法之一。

### 14. DDPG
- 结构：Deterministic Actor + Critic + Replay Buffer + Target Network。
- 目标：学习确定性策略并用 Q 函数指导策略改进。
- 特性：适合连续动作、off-policy、样本效率高于 PPO；但对超参数和探索噪声较敏感。

### 15. SAC
- 结构：随机策略 Actor + 双 Q Critic + 熵正则项 + Replay Buffer。
- 目标：最大化期望回报的同时最大化策略熵，即兼顾收益和探索。
- 特性：稳定、鲁棒、样本效率高，是连续控制中非常强的通用算法；通常比 DDPG 更稳。

## 一条理解主线

这组算法的演化逻辑可以概括为：

- PG：能直接学策略，但方差大。
- Actor-Critic：加入 Critic 降低方差。
- TRPO/PPO：进一步约束策略更新，提升稳定性。
- DDPG/SAC：把思路扩展到连续控制和 off-policy 设定，进一步提高样本效率。

## 进阶范式与特殊任务设定

### 16. 模仿学习 IL
- 结构：通常由专家数据 + 策略网络组成，典型方法是 Behavior Cloning。
- 目标：直接拟合专家策略，绕开环境奖励设计和大量在线试错。
- 特性：训练简单、起步快；但容易受到分布偏移影响，错误会在连续决策中累积。

### 17. 最大边际 IRL
- 结构：通过专家轨迹反推奖励函数，并要求专家行为相对其他策略具有边际优势。
- 目标：学习一个能解释专家行为优越性的奖励函数。
- 特性：强调区分性；适合从专家行为中恢复任务偏好，但优化往往较复杂。

### 18. 最大熵 IRL
- 结构：在 IRL 中加入熵最大化原则，使专家轨迹概率与奖励成指数关系。
- 目标：在解释专家行为的同时保留行为随机性，避免过度确定化。
- 特性：理论更自然，对多模态专家行为更友好，是现代 IRL 的重要路线。

### 19. 模型预测控制 MPC
- 结构：环境模型 + 有限时域滚动优化 + 执行第一步动作。
- 目标：在每个时刻求解短期最优控制序列，实时闭环控制系统。
- 特性：强依赖模型质量；在控制领域非常常见，适合安全和约束明确的问题。

### 20. 基于模型的策略优化
- 结构：先学习环境动态模型，再利用模型生成数据或直接优化策略。
- 目标：减少真实环境交互成本，提高样本效率。
- 特性：当模型准确时很高效；但模型误差会累积并导致策略偏移。

### 21. 离线强化学习 Offline RL
- 结构：只使用固定数据集训练，不允许在线与环境交互。
- 目标：从历史数据中学习高质量策略，同时避免分布外动作导致的价值误估。
- 特性：适合医疗、推荐、自动驾驶等难以在线探索的场景；核心难点是 distribution shift。

### 22. 目标导向强化学习 Goal-Conditioned RL
- 结构：策略和价值函数显式接收目标 $g$ 作为输入。
- 目标：学习“给定目标时该如何行动”，而不是只优化单一任务奖励。
- 特性：适合稀疏奖励和多任务问题，HER 等方法常与其结合。

### 23. 多智能体强化学习 MARL
- 结构：多个 agent 同时学习，常见范式是独立学习或集中训练、分散执行 CTDE。
- 目标：在合作、竞争或混合博弈中学习联合行为策略。
- 特性：环境非平稳、信用分配困难、维度爆炸明显，是 RL 中最复杂的方向之一。

### 24. 多智能体强化学习进阶
- 结构：在 MARL 上进一步引入价值分解、集中式 critic、博弈论建模或通信机制。
- 目标：提升协作质量、训练稳定性与可扩展性。
- 特性：代表性思路包括 VDN、QMIX、MADDPG 等，适合复杂协作或对抗场景。

## 各类算法对比表

### 主流单智能体算法对比

| 算法 | 结构 | 优化目标 | 数据方式 | 动作空间 | 主要特性 | 主要局限 |
| --- | --- | --- | --- | --- | --- | --- |
| DP | 价值迭代/策略迭代 + 已知模型 | Bellman 最优方程 | 基于模型 | 离散为主 | 理论清晰，能求精确解 | 需要已知模型 |
| MC | 轨迹采样 + 回报平均 | 回合回报估计 | On-policy 常见 | 离散/连续均可 | 无偏，概念直观 | 方差大，必须等回合结束 |
| TD | bootstrap 价值更新 | TD 误差最小化 | 在线采样 | 离散/连续均可 | 更新及时，效率高 | 有偏差 |
| Dyna-Q | 真实交互 + 模型规划 | 价值优化 + 规划改进 | 在线 + 模拟 | 离散为主 | 样本效率更高 | 依赖模型质量 |
| DQN | Q 网络 + 回放 + 目标网络 | TD/Bellman 误差 | Off-policy | 离散 | 能处理高维状态 | 不适合连续动作 |
| Double/Dueling/PER | DQN 增强模块 | 更稳定的 Q 学习 | Off-policy | 离散 | 降低过估计，提升表达能力 | 本质上仍受 DQN 框架限制 |
| PG | 直接策略网络 | 最大化期望回报 | On-policy | 离散/连续 | 直接优化策略，适用连续控制 | 方差大，样本效率低 |
| Actor-Critic | Actor + Critic | 策略目标 + 价值目标 | 多为 On-policy，也可 Off-policy | 离散/连续 | 现代策略算法基础模板 | 训练耦合更复杂 |
| TRPO | Actor-Critic + 信赖域 | 受约束策略优化 | On-policy | 离散/连续 | 更新稳定，理论较强 | 实现复杂，计算量大 |
| PPO | Actor-Critic + Clip | 近端策略优化 | On-policy | 离散/连续 | 简洁稳健，应用最广 | 样本效率一般 |
| DDPG | 确定性 Actor + Critic | Q 指导的策略优化 | Off-policy | 连续 | 样本效率较高 | 稳定性一般，调参敏感 |
| SAC | 随机 Actor + 双 Q + 熵项 | 回报 + 熵最大化 | Off-policy | 连续 | 稳定、鲁棒、探索强 | 结构较复杂 |
| MPC | 模型 + 在线优化器 | 有限时域最优控制 | 基于模型 | 连续常见 | 可处理约束，控制解释性强 | 求解成本高，依赖模型 |
| 基于模型策略优化 | 动力学模型 + 策略优化 | 用模型辅助策略学习 | 模型生成数据 | 离散/连续 | 样本效率高 | 模型误差会传递到策略 |
| Offline RL | 固定数据集 + 保守学习机制 | 离线数据上的策略优化 | 离线 | 离散/连续 | 无需在线试错 | 分布外动作估值困难 |

### 特殊设定与扩展方向对比

| 方法 | 结构 | 目标 | 特性 |
| --- | --- | --- | --- |
| 多臂老虎机 | 动作价值估计 + 探索策略 | 最大化累计奖励 | 没有状态转移，是探索研究起点 |
| MDP | 状态、动作、转移、奖励、折扣因子 | 定义序列决策问题 | 是统一框架，不是具体算法 |
| MCTS | 树搜索 + 模拟 | 在线规划和决策 | 适合可模拟环境 |
| 模仿学习 | 专家数据 + 策略拟合 | 模仿专家行为 | 快速起步，但会受分布偏移影响 |
| 最大边际 IRL | 奖励学习 + 边际约束 | 恢复可解释奖励 | 区分性强，优化复杂 |
| 最大熵 IRL | 奖励学习 + 熵正则 | 恢复奖励并保留随机性 | 适合多模态专家行为 |
| 目标导向 RL | 策略/价值输入目标 | 达成指定目标 | 适合稀疏奖励、多任务 |
| 多智能体 RL | 多 agent 联合学习 | 学习合作或竞争策略 | 非平稳、信用分配难 |
| 多智能体进阶 | CTDE、价值分解、通信 | 提升协作与扩展性 | 更适合复杂协同场景 |

## 总结

如果只抓住一条主线，可以这样记：

- 想理解 RL 基础，从 MDP、DP、MC、TD 开始。
- 想做离散动作深度强化学习，从 DQN 及其改进开始。
- 想做通用策略优化，从 Actor-Critic、PPO 开始。
- 想做连续控制，优先关注 DDPG 和 SAC，其中 SAC 往往更稳。
- 想提高样本效率，关注基于模型方法和离线强化学习。
- 想处理复杂任务设定，进一步看目标导向、多智能体和模仿学习。